# Fleet Simulation & Monitoring · Edge-Computing Notebook

Everything so far ran on **one** edge node. Real deployments have many: a fleet of devices in vehicles, buildings, or factories. At scale, new problems appear: you cannot watch each device by hand, devices fail silently, and you need to know at a glance which ones are healthy.

In this capstone you will **simulate a fleet** of devices that each publish telemetry over MQTT, build a **central monitor** that tracks every device's health, detect a device that goes offline, and practice **scaling** the fleet up and down.

```text
Many simulated devices --MQTT--> Broker --MQTT--> Fleet Monitor (live status table)
```

You will:

- build a **simulated device** image whose identity comes from its container hostname
- build a **fleet monitor** that subscribes to every device with a wildcard topic
- wire broker + devices + monitor together with **Compose** and launch 5 replicas
- **scale to 10** devices with one command and no code changes
- **kill a device** and watch the monitor flag it OFFLINE, then bring it back

The main idea: managing one device is programming; managing a fleet is **operations**. Identity, monitoring, failure detection, and orchestration become the hard parts.

## How this notebook works · cells and a Jupyter terminal

Steps that run once and return (writing files, `docker compose up -d`, scaling) are ordinary **notebook cells** · just run them here.

Watching the monitor's live status stream never returns on its own, and the failure drill in Part 7 needs you to **watch the monitor while stopping a device** · that takes **two terminals side by side**. For those, this notebook **writes small scripts into your lab folder and asks you to run them in a Jupyter terminal.**

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal (**File ▸ New ▸ Terminal**, or the Terminal tile in the Launcher) and run the given command there. You can open several terminals at once.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell writes `~/fleetLab/labEnv.sh`, and every helper script starts with `source ~/fleetLab/labEnv.sh` · notebook and terminals agree on your `USER` and Compose project name with no manual step. The fleet itself talks over an internal Compose network, so this lab needs **no host ports** and cannot collide with other students.

**Requirements:** a Python 3 kernel on the Jetson Thor, Docker + Compose available to your user. This lab builds on the MQTT pattern from the *MQTT / HTTP / WebSocket* lab.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **Jetson Thor** edge device · and this notebook runs there too. Your JupyterLab session lives on the Thor, so **every cell in this exercise runs on the Thor**, not on your own computer.

Because the Thor is shared:

- use your own home folder and keep your files under it
- prefix Docker resources and the Compose project with your account name using `${USER}` (the cells below already do)
- the whole fleet here runs as containers **inside one Compose project**, talking over an internal network, so it needs no host ports and will not collide with other students

---
## Part 1 · Set Up the Fleet Project

**[Notebook cell]** The cell below creates `~/fleetLab`, writes `labEnv.sh`, and exports `FLEET_PROJECT` · the `${USER}-fleet` Compose project name used everywhere. (The `PORT` it derives is a spare; this lab publishes no host ports.)

In [ ]:
import os

# No host ports are needed - the fleet talks over an internal Compose network.
# FLEET_PROJECT names your Compose project so it never collides with others.
labConfig = setupLab(
    "fleetLab",
    ports={"PORT": 5000},   # spare; unused by the fleet
    extraEnv={"FLEET_PROJECT": f"{os.environ.get('USER', 'student01')}-fleet"},
)

### Preflight · check your environment

Run this once at the start and fix any failing check before continuing.

In [ ]:
import os

preflight(
    [
        check("docker daemon reachable", dockerDaemonUp(),
              hint="Docker must be running and your account must have an instructor-provisioned rootless Podman socket. Try "
                   "<code>docker info</code> in a Terminal; if it fails, ask your instructor.",
              link="https://docs.docker.com/engine/daemon/troubleshoot/",
              linkText="Docker daemon troubleshooting"),
        check("docker compose available", composeAvailable(),
              hint="The fleet is wired together with Compose. If <code>docker compose version</code> "
                   "fails in a Terminal, ask your instructor.",
              link="https://docs.docker.com/compose/", linkText="Docker Compose docs"),
    ],
    infoRows=[("your USER", os.environ.get("USER", "?")),
              ("your Compose project", os.environ.get("FLEET_PROJECT", "?"))],
)

**[Notebook cell]** Create the project structure:

In [ ]:
!mkdir -p ~/fleetLab/device ~/fleetLab/monitor ~/fleetLab/mosquitto
%cd ~/fleetLab

The structure will become:

```text
fleetLab/
  mosquitto/
    mosquitto.conf
  device/
    publisher.py
    Dockerfile
  monitor/
    fleetMonitor.py
    Dockerfile
  compose.yaml
```

---
## Part 2 · From One Device to a Fleet

One device is easy to reason about. A fleet changes what matters:

```text
One device                     A fleet
-----------                    -------
you watch its logs             you cannot watch hundreds of logs
you know it's running          devices fail silently; you need heartbeats
you ssh in to update           you need to update many at once
one identity                   every device needs a unique identity
```

The pattern that scales: each device **publishes** telemetry tagged with its own identity to a broker, and a central **monitor** subscribes to all of them. Devices do not need to know about each other or about the monitor · only about the broker and the topic.

---
## Part 3 · Build the Simulated Device

Each device publishes a reading every couple of seconds, tagged with a unique identity. The identity defaults to the container's hostname, which Docker makes unique per replica · so one image becomes a whole fleet.

**[Notebook cell]** Create `device/publisher.py` and `device/Dockerfile`:

In [ ]:
%%writefile device/publisher.py
import os
import json
import time
import random
import socket
from datetime import datetime, timezone
import paho.mqtt.client as mqtt

# unique per container: each scaled replica has its own hostname
deviceID = os.environ.get("DEVICE_ID") or socket.gethostname()
brokerHost = os.environ.get("BROKER", "broker")
topic = f"edge/{deviceID}/telemetry"

client = mqtt.Client()
client.connect(brokerHost, 1883, 60)
client.loop_start()
print(f"device {deviceID} publishing to {topic}", flush=True)

while True:
    reading = {
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "device_id": deviceID,
        "temperature": round(random.uniform(22, 40), 2),
        "battery": random.randint(20, 100),
    }
    client.publish(topic, json.dumps(reading))
    time.sleep(2)

In [ ]:
%%writefile device/Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN pip install "paho-mqtt<2.0"

COPY publisher.py .

CMD ["python", "publisher.py"]

In [ ]:
# Preview the device we just wrote.
showFile("~/fleetLab/device/publisher.py", language="python")
showFile("~/fleetLab/device/Dockerfile", language="docker")

### Checkpoint · Part 3

In [ ]:
checkpoint(
    "Part 3 - simulated device built",
    [
        check("publisher.py identity comes from the hostname",
              fileContains("~/fleetLab/device/publisher.py", "socket.gethostname()"),
              hint="Re-run the %%writefile device/publisher.py cell in Part 3 - the hostname "
                   "fallback is what makes every scaled replica unique.",
              link="https://docs.python.org/3/library/socket.html#socket.gethostname",
              linkText="socket.gethostname"),
        check("publisher tags readings with device_id",
              fileContains("~/fleetLab/device/publisher.py", "\"device_id\": deviceID"),
              hint="Re-run the %%writefile device/publisher.py cell in Part 3.",
              link="https://mqtt.org/", linkText="MQTT.org"),
        check("device Dockerfile pins paho-mqtt below 2.0",
              fileContains("~/fleetLab/device/Dockerfile", "paho-mqtt<2.0"),
              hint="Re-run the %%writefile device/Dockerfile cell in Part 3.",
              link="https://docs.docker.com/reference/dockerfile/", linkText="Dockerfile reference"),
    ],
    successNote="One image, any number of devices - identity solved by the container hostname.",
    docLink="https://docs.docker.com/reference/dockerfile/",
    docLinkText="Dockerfile reference",
)

---
## Part 4 · Build the Fleet Monitor

The monitor subscribes to **every** device's topic with a wildcard, records when each device was last heard from, and prints a live status table. A device that stops reporting is flagged **OFFLINE**.

**[Notebook cell]** Create `monitor/fleetMonitor.py` and `monitor/Dockerfile`:

In [ ]:
%%writefile monitor/fleetMonitor.py
import os
import json
import time
import threading
import paho.mqtt.client as mqtt

brokerHost = os.environ.get("BROKER", "broker")
offlineAfter = float(os.environ.get("OFFLINE_AFTER", "8"))  # seconds without a message

devices = {}            # deviceID -> {"lastSeen": epoch, "reading": {...}}
lock = threading.Lock()


def onConnect(client, userData, flags, rc):
    client.subscribe("edge/+/telemetry")  # + matches every device id
    print(f"fleet monitor connected (rc={rc}); watching edge/+/telemetry", flush=True)


def onMessage(client, userData, message):
    try:
        data = json.loads(message.payload.decode())
    except ValueError:
        return
    deviceID = data.get("device_id", message.topic)
    with lock:
        devices[deviceID] = {"lastSeen": time.time(), "reading": data}


def printStatus():
    while True:
        time.sleep(5)
        now = time.time()
        with lock:
            print("\n==== FLEET STATUS ====", flush=True)
            print(f"{'device':<26}{'status':<10}{'age(s)':<8}{'temp':<7}battery", flush=True)
            onlineCount = 0
            for deviceID in sorted(devices):
                info = devices[deviceID]
                age = now - info["lastSeen"]
                if age <= offlineAfter:
                    status = "ONLINE"
                    onlineCount += 1
                else:
                    status = "OFFLINE"
                reading = info["reading"]
                print(f"{deviceID:<26}{status:<10}{age:<8.1f}"
                      f"{str(reading.get('temperature')):<7}{reading.get('battery')}", flush=True)
            print(f"-- {onlineCount}/{len(devices)} devices online --", flush=True)


client = mqtt.Client()
client.on_connect = onConnect
client.on_message = onMessage
client.connect(brokerHost, 1883, 60)

threading.Thread(target=printStatus, daemon=True).start()
client.loop_forever()

In [ ]:
%%writefile monitor/Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN pip install "paho-mqtt<2.0"

COPY fleetMonitor.py .

CMD ["python", "fleetMonitor.py"]

In [ ]:
# Preview the monitor we just wrote.
showFile("~/fleetLab/monitor/fleetMonitor.py", language="python")
showFile("~/fleetLab/monitor/Dockerfile", language="docker")

### Checkpoint · Part 4

In [ ]:
checkpoint(
    "Part 4 - fleet monitor built",
    [
        check("monitor subscribes with the + wildcard",
              fileContains("~/fleetLab/monitor/fleetMonitor.py", "edge/+/telemetry"),
              hint="Re-run the %%writefile monitor/fleetMonitor.py cell in Part 4 - the wildcard "
                   "is what lets one subscription cover the whole fleet.",
              link="https://mosquitto.org/man/mqtt-7.html",
              linkText="MQTT topics and wildcards"),
        check("monitor flags stale devices OFFLINE",
              fileContains("~/fleetLab/monitor/fleetMonitor.py", "OFFLINE"),
              hint="Re-run the %%writefile monitor/fleetMonitor.py cell in Part 4.",
              link="https://mqtt.org/", linkText="MQTT.org"),
        check("monitor Dockerfile copies fleetMonitor.py",
              fileContains("~/fleetLab/monitor/Dockerfile", "fleetMonitor.py"),
              hint="Re-run the %%writefile monitor/Dockerfile cell in Part 4.",
              link="https://docs.docker.com/reference/dockerfile/", linkText="Dockerfile reference"),
    ],
    successNote="One subscriber now watches every device that will ever join the fleet.",
    docLink="https://mosquitto.org/man/mqtt-7.html",
    docLinkText="MQTT topics and wildcards",
)

---
## Part 5 · Wire the Fleet Together with Compose

**[Notebook cell]** Create the broker config and `compose.yaml`:

In [ ]:
%%writefile mosquitto/mosquitto.conf
listener 1883
allow_anonymous true

In [ ]:
%%writefile compose.yaml
services:
  broker:
    image: eclipse-mosquitto
    volumes:
      - ./mosquitto/mosquitto.conf:/mosquitto/config/mosquitto.conf

  device:
    build: ./device
    depends_on:
      - broker
    environment:
      BROKER: broker
    restart: unless-stopped

  monitor:
    build: ./monitor
    depends_on:
      - broker
    restart: unless-stopped
    environment:
      BROKER: broker
      OFFLINE_AFTER: 8

In [ ]:
# Preview the compose file we just wrote.
showFile("~/fleetLab/compose.yaml", language="yaml")

Notice the `device` service has no fixed name or port · that is what lets it scale to many replicas.

> **Why `restart: unless-stopped`:** `depends_on` waits for the broker *container* to start, not for it to be *ready*. A service that connects a moment too early exits · the restart policy brings it right back to reconnect, so a startup race doesn't leave you with a dead monitor.

### Checkpoint · Part 5

In [ ]:
import os
userName = os.environ.get("USER", "student01")

checkpoint(
    "Part 5 - fleet wired with Compose",
    [
        check("broker config written",
              fileContains("~/fleetLab/mosquitto/mosquitto.conf", "allow_anonymous true"),
              hint="Re-run the %%writefile mosquitto/mosquitto.conf cell in Part 5.",
              link="https://mosquitto.org/man/mosquitto-conf-5.html",
              linkText="mosquitto.conf manual"),
        check("compose.yaml survives startup races",
              fileContains("~/fleetLab/compose.yaml", "restart: unless-stopped"),
              hint="Re-run the %%writefile compose.yaml cell in Part 5 - both device and monitor "
                   "need <code>restart: unless-stopped</code>.",
              link="https://docs.docker.com/engine/containers/start-containers-automatically/",
              linkText="Restart policies"),
        check("compose file is valid",
              commandSucceeds(f"cd ~/fleetLab && docker compose -p {userName}-fleet config --quiet"),
              hint="Run <code>docker compose config</code> in a Terminal from ~/fleetLab to see "
                   "the YAML error, fix the compose cell, and re-run it.",
              link="https://docs.docker.com/reference/compose-file/",
              linkText="Compose file reference"),
    ],
    successNote="Broker, devices, and monitor described in one file - ready to launch.",
    docLink="https://docs.docker.com/reference/compose-file/",
    docLinkText="Compose file reference",
)

---
## Part 6 · Launch and Scale the Fleet

**[Notebook cell]** Start the fleet with **5** simulated devices, using your account name as the project name. `-d` detaches, so the cell returns; the first run also builds both images (expect a minute or two):

In [ ]:
!docker compose -p ${USER}-fleet up --build --scale device=5 -d

**[Notebook cell]** Confirm five devices plus a broker and a monitor are running:

In [ ]:
!docker compose -p ${USER}-fleet ps

**You should see:** seven containers `Up` · one `broker`, one `monitor`, and five `device` replicas (their names end in `-1` through `-5`).

> **If it doesn't work:** if a container shows `Restarting` or `Exited`, check its logs, e.g. `docker compose -p ${USER}-fleet logs monitor`. A brief restart right after `up` is the normal broker-readiness race described above and resolves itself. If the build fails, re-check that `device/Dockerfile` and `monitor/Dockerfile` exist and that the notebook is in `~/fleetLab` (the `%cd` cell in Part 1).

**[Notebook cell]** Write the live-monitor helper:

In [ ]:
%%writefile watchMonitor.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/fleetLab/labEnv.sh
cd ~/fleetLab

echo "Live fleet status from the monitor. Press Ctrl-C to stop watching (the fleet keeps running)."
docker compose -p "${USER}-fleet" logs -f monitor

In [ ]:
!chmod +x watchMonitor.sh

**[Terminal]** Watch the live fleet status from the monitor (it streams forever, so it belongs in a terminal):

```bash
~/fleetLab/watchMonitor.sh
```

**You should see:** every ~5 seconds a `==== FLEET STATUS ====` table, one row per device (`ONLINE`, a small age, latest temp/battery), ending with `-- 5/5 devices online --`. Device ids are random container hostnames, not `sensor01`-style names. Stop watching with **Ctrl-C** (the fleet keeps running).

> **New term · heartbeat:** a regular "I'm alive" signal. Here each device's telemetry message doubles as its heartbeat; the monitor marks a device OFFLINE when it hasn't heard one within `OFFLINE_AFTER` seconds.

**[Notebook cell]** Now **scale up** to 10 devices with no code changes:

In [ ]:
!docker compose -p ${USER}-fleet up --scale device=10 -d
!docker compose -p ${USER}-fleet ps

Watch the monitor again in the **[Terminal]** (`~/fleetLab/watchMonitor.sh`) and confirm new devices appear · the table grows to `-- 10/10 devices online --`.

One image, one command, a bigger fleet. This is the core idea of edge orchestration: scale the number of running instances without touching the application.

### Checkpoint · Part 6

In [ ]:
import os
userName = os.environ.get("USER", "student01")

def deviceReplicas(minCount):
    """Probe: count running containers named <user>-fleet-device-*."""
    def probe():
        namesOut, _ = runShell(["docker", "ps", "--format", "{{.Names}}"])
        replicaNames = [name for name in namesOut.splitlines()
                        if name.startswith(f"{userName}-fleet-device")]
        return len(replicaNames) >= minCount, f"{len(replicaNames)} device replica(s) running"
    return probe

checkpoint(
    "Part 6 - fleet launched and scaled",
    [
        check("broker is running",
              containerRunning(f"{userName}-fleet-broker-1"),
              hint="Re-run the compose up cell in Part 6, then check "
                   "<code>docker compose -p <you>-fleet logs broker</code> in a Terminal.",
              link="https://docs.docker.com/reference/cli/docker/compose/up/",
              linkText="docker compose up reference"),
        check("monitor is running",
              containerRunning(f"{userName}-fleet-monitor-1"),
              hint="A brief restart right after up is the normal broker-readiness race and heals "
                   "itself - wait ~15s and re-run this checkpoint. Still down? Check "
                   "<code>docker compose -p <you>-fleet logs monitor</code>.",
              link="https://docs.docker.com/engine/containers/start-containers-automatically/",
              linkText="Restart policies"),
        check("at least 10 device replicas are running",
              deviceReplicas(10),
              hint="Re-run the scale cell in Part 6 "
                   "(<code>docker compose -p <you>-fleet up --scale device=10 -d</code>).",
              link="https://docs.docker.com/reference/cli/docker/compose/up/",
              linkText="docker compose up --scale"),
        check("monitor is printing fleet status tables",
              outputContains(f"cd ~/fleetLab && docker compose -p {userName}-fleet logs --tail 100 monitor",
                             "FLEET STATUS"),
              hint="Give the monitor ~10 seconds after startup to print its first table, then "
                   "re-run this checkpoint. If it never prints, check the monitor logs.",
              link="https://docs.docker.com/reference/cli/docker/compose/logs/",
              linkText="docker compose logs reference"),
    ],
    successNote="A 10-device fleet from one image and one command - that is orchestration.",
    docLink="https://docs.docker.com/reference/cli/docker/compose/up/",
    docLinkText="docker compose up reference",
)

---
## Part 7 · Detect a Failed Device

The point of monitoring is catching failures. Make one device fail · while you watch the monitor notice. This step genuinely needs **two terminals side by side**.

**[Notebook cell]** List the running device containers, then write the two control scripts:

In [ ]:
!docker compose -p ${USER}-fleet ps device

In [ ]:
%%writefile stopOneDevice.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/fleetLab/labEnv.sh
cd ~/fleetLab

victim=$(docker ps --format '{{.Names}}' | grep "^${USER}-fleet-device" | head -n 1)
if [ -z "${victim}" ]; then
  echo "no running device replica found - is the fleet up?"
  exit 1
fi
echo "stopping ${victim} ..."
docker stop "${victim}"
echo "now watch the monitor terminal: within ~8s this device flips to OFFLINE."

In [ ]:
%%writefile startDevices.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/fleetLab/labEnv.sh
cd ~/fleetLab

echo "starting any stopped device replicas ..."
docker compose -p "${USER}-fleet" start device

In [ ]:
!chmod +x stopOneDevice.sh startDevices.sh

**[Terminal 1]** Watch the live fleet status and keep it open:

```bash
~/fleetLab/watchMonitor.sh
```

**[Terminal 2]** Stop a single device, then look at Terminal 1:

```bash
~/fleetLab/stopOneDevice.sh
```

**You should see:** within `OFFLINE_AFTER` seconds (8 by default), the stopped device's row flips from `ONLINE` to `OFFLINE`, its age keeps climbing, and the summary line drops (e.g. `-- 9/10 devices online --`) while every other device stays online. The monitor caught a silent failure with nobody watching individual logs: the heartbeat (last-seen time) did the work.

> Because the `device` service uses `restart: unless-stopped`, a device that **crashes** is restarted automatically. You used `docker stop`, which counts as intentional, so it stays down until you restart it · letting you observe the OFFLINE state.

**[Terminal 2]** Bring it back and watch Terminal 1 again:

```bash
~/fleetLab/startDevices.sh
```

The monitor returns it to **ONLINE** once readings resume. This is the detect-and-recover loop that keeps a fleet healthy.

### Checkpoint · Part 7

In [ ]:
import os
userName = os.environ.get("USER", "student01")

def deviceReplicas(minCount):
    """Probe: count running containers named <user>-fleet-device-*."""
    def probe():
        namesOut, _ = runShell(["docker", "ps", "--format", "{{.Names}}"])
        replicaNames = [name for name in namesOut.splitlines()
                        if name.startswith(f"{userName}-fleet-device")]
        return len(replicaNames) >= minCount, f"{len(replicaNames)} device replica(s) running"
    return probe

checkpoint(
    "Part 7 - failure detected and recovered",
    [
        check("monitor logged an OFFLINE device",
              outputContains(f"cd ~/fleetLab && docker compose -p {userName}-fleet logs --tail 400 monitor",
                             "OFFLINE"),
              hint="Run <code>~/fleetLab/stopOneDevice.sh</code> in a Terminal, wait ~15 seconds "
                   "for the monitor to flag it, then re-run this checkpoint.",
              link="https://docs.docker.com/reference/cli/docker/compose/logs/",
              linkText="docker compose logs reference"),
        check("fleet recovered to 10 running devices",
              deviceReplicas(10),
              hint="Run <code>~/fleetLab/startDevices.sh</code> in a Terminal to restart the "
                   "stopped replica, then re-run this checkpoint.",
              link="https://docs.docker.com/reference/cli/docker/compose/start/",
              linkText="docker compose start reference"),
        check("monitor survived the whole drill",
              containerRunning(f"{userName}-fleet-monitor-1"),
              hint="Re-run the compose up cell in Part 6 and check the monitor logs.",
              link="https://docs.docker.com/engine/containers/start-containers-automatically/",
              linkText="Restart policies"),
    ],
    successNote="Silent failure caught by a heartbeat, recovery confirmed - the core loop of "
                "fleet operations.",
    docLink="https://docs.docker.com/reference/cli/docker/compose/",
    docLinkText="docker compose CLI reference",
)

---
## Part 8 · Orchestration Concepts

You just used the basic levers of fleet orchestration:

```text
Scaling            -> run N instances of one service (--scale device=N)
Health / heartbeat -> devices report regularly; missing reports mean trouble
Restart policy     -> crashed instances recover automatically (restart: unless-stopped)
Identity           -> every instance is uniquely addressable (its own topic)
```

Real fleet platforms (Kubernetes, k3s, AWS IoT, Azure IoT Hub, Balena) add more · rolling updates, where devices are upgraded a few at a time so the fleet never fully goes down; placement, deciding which workload runs on which device; and remote configuration. The patterns are the same ones you built here, scaled to thousands of devices across many physical locations.

Optional extension: point the fleet monitor at the **InfluxDB + Grafana** stack from the *Grafana & Time-Series Database* lab · write each reading with `device_id` as a tag, then build a Grafana panel grouped by device for a true fleet dashboard with history and alerts.

---
## Cleanup

**[Notebook cell]** Stop and remove the entire fleet, including its network and volumes, then confirm nothing of yours is left running (safe to run any number of times):

In [ ]:
%cd ~/fleetLab
!docker compose -p ${USER}-fleet down -v
!docker compose -p ${USER}-fleet ps

**[Notebook cell]** Optional · remove your lab folder (uncomment to run). Only remove your own files, containers, networks, and volumes.

In [ ]:
# !rm -rf ~/fleetLab

### Lab scorecard

Run this any time for a live summary of every checkpoint in this kernel session. If a row is incomplete, scroll up, fix that part, re-run its checkpoint, and re-run this cell.

In [ ]:
labSummary("Fleet Simulation & Monitoring")

---
## Connect to the Rest of the Course

This capstone combined nearly everything you built:

```text
Docker + Compose       -> run and scale the fleet
Telemetry format       -> what each device reports
MQTT pub/sub           -> how devices and the monitor communicate
Monitoring + heartbeat -> detect failure across many devices
Orchestration          -> scale, restart, identity
(optional) TSDB + Grafana -> store and visualize the whole fleet
```

A real project runs fleets like this for connected vehicles, smart-building sensors, retail edge nodes, industrial machines, and robotics. The main idea: edge computing rarely stops at one device · the discipline is keeping a whole fleet observable, recoverable, and scalable, which is exactly the system you simulated here.

---
## Key Terms

- **Fleet:** many edge devices managed together as one system.
- **Orchestration:** running, scaling, updating, and recovering many containers/devices as a group rather than one at a time.
- **Scaling (`--scale device=N`):** running N copies of one service from a single image and command.
- **Heartbeat:** a regular "I'm alive" signal; a missing heartbeat is how you detect a silent failure.
- **Restart policy (`restart: unless-stopped`):** the rule that automatically restarts a crashed container so it reconnects and recovers.
- **Device identity:** a unique name per device (here the container hostname) so its data and status can be tracked separately.
- **Rolling update:** upgrading a fleet a few devices at a time so the whole system never goes down at once.